In [ ]:
# ======================================================================
# 【研究环境专享】TCN + Temporal Attention 训练全管线 (究极版)
# ======================================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import weight_norm
import os
import gc
from jqdata import *
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. 全局参数设置
# ---------------------------------------------------------
START_YEAR = 2018
END_YEAR = 2023
SEQ_LENGTH = 30           # 历史观察窗口 (30天)
FUTURE_DAYS = 5           # 预测未来收益窗口 (5天)
FEATURE_COLS = ['open', 'close', 'high', 'low', 'volume', 'money']

EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 0.001

# ---------------------------------------------------------
# 2. 数据按年分块清洗与落盘 (严格对齐小市值选股池)
# ---------------------------------------------------------
def process_and_save_yearly_data(year):
    print(f"🚀 开始处理 {year} 年的切片数据...")
    tdays = get_trade_days(start_date=f'{year}-01-01', end_date=f'{year}-12-31')
    sample_days = tdays[::5] # 约每周采样一次截面
    all_trade_days = list(get_all_trade_days())
    
    X_year, y_year = [], []
    valid_samples = 0
    
    for date in sample_days:
        # 提取当天的目标选股池 (ROE>5, ROA>3, 市值最小150只)
        q = query(valuation.code).filter(
            indicator.roe > 5.0, indicator.roa > 3.0
        ).order_by(valuation.market_cap.asc()).limit(150)
        
        try:
            pool = list(get_fundamentals(q, date=date)['code'])
        except: continue
        if not pool: continue

        # 获取历史量价张量 (过去30天，截断在 date 当天)
        hist_df = get_price(pool, end_date=date, count=SEQ_LENGTH, 
                            fields=FEATURE_COLS, frequency='daily', panel=False)
        if hist_df.empty: continue
        
        # 获取未来绝对收益 (未来5天，使用 start_date + end_date 组合规避冲突)
        current_idx = all_trade_days.index(date)
        if current_idx + FUTURE_DAYS >= len(all_trade_days): continue
        future_date = all_trade_days[current_idx + FUTURE_DAYS]

        fut_df = get_price(pool, start_date=date, end_date=future_date, 
                           fields=['close'], frequency='daily', panel=False)
        if fut_df.empty: continue

        # 截面打打标与特征归一化
        future_rets, hist_tensors = {}, {}
        for stock in pool:
            s_hist = hist_df[hist_df['code'] == stock]
            if len(s_hist) < SEQ_LENGTH: continue
            
            data_mat = s_hist[FEATURE_COLS].values
            mean = np.mean(data_mat, axis=0, keepdims=True)
            std = np.std(data_mat, axis=0, keepdims=True) + 1e-8
            hist_tensors[stock] = (data_mat - mean) / std
            
            s_fut = fut_df[fut_df['code'] == stock]
            if len(s_fut) < FUTURE_DAYS + 1: continue
            future_rets[stock] = s_fut.iloc[-1]['close'] / s_fut.iloc[0]['close'] - 1
            
        if not future_rets: continue
        
        # 截面排名打标 (前30%的股票标记为1，其余为0)
        labels = (pd.Series(future_rets).rank(pct=True) >= 0.70).astype(float)
        
        for stock in labels.index:
            X_year.append(hist_tensors[stock])
            y_year.append(labels[stock])
            valid_samples += 1

    if valid_samples > 0:
        X_np = np.transpose(np.array(X_year), (0, 2, 1)) # (Batch, Features, Seq)
        y_np = np.array(y_year).reshape(-1, 1)
        torch.save((torch.tensor(X_np, dtype=torch.float32), torch.tensor(y_np, dtype=torch.float32)), f'tcn_attn_chunk_{year}.pt')
        print(f"✅ {year} 年完成! 提取纯正小市值样本 {valid_samples} 个，已成功序列化落盘。")
    
    del X_year, y_year; gc.collect()

# 执行离线分块清洗
for y in range(START_YEAR, END_YEAR + 1): 
    process_and_save_yearly_data(y)

# ---------------------------------------------------------
# 3. 神经网络架构定义 (TCN + Temporal Attention)
# ---------------------------------------------------------
class ChainedCausalConv(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(ChainedCausalConv, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = nn.ConstantPad1d((-padding, 0), 0)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = nn.ConstantPad1d((-padding, 0), 0)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.net(x) + (x if self.downsample is None else self.downsample(x)))

class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super(TemporalAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        x_transposed = x.transpose(1, 2) # (Batch, Seq_Length, Channels)
        attn_weights = self.attention(x_transposed)
        attn_weights = torch.softmax(attn_weights, dim=1) # 对时间轴进行归一化
        context = torch.sum(x_transposed * attn_weights, dim=1) # 动态加权求和
        return context, attn_weights

class TCNModel(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=3, dropout=0.2):
        super(TCNModel, self).__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            layers += [ChainedCausalConv(in_channels, num_channels[i], kernel_size, stride=1, dilation=dilation_size, padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.tcn = nn.Sequential(*layers)
        self.attention = TemporalAttention(num_channels[-1])
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        tcn_out = self.tcn(x)
        context, attn_weights = self.attention(tcn_out)
        return self.linear(context)

# ---------------------------------------------------------
# 4. 离线流式训练循环 (搭载双重防线)
# ---------------------------------------------------------
model = TCNModel(input_size=len(FEATURE_COLS), output_size=1, num_channels=[16, 32, 64])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

chunk_files = sorted([f for f in os.listdir('.') if f.startswith('tcn_attn_chunk_') and f.endswith('.pt')])
print(f"\n🧠 找到 {len(chunk_files)} 个小市值专属特征文件，开启注意力网络训练...")

model.train()
for epoch in range(EPOCHS):
    total_loss, correct_preds, total_samples, valid_batches = 0, 0, 0, 0
    for file in chunk_files:
        X_chunk, y_chunk = torch.load(file)
        dataloader = DataLoader(TensorDataset(X_chunk, y_chunk), batch_size=BATCH_SIZE, shuffle=True)
        
        for batch_X, batch_y in dataloader:
            # 防线一：脏数据彻底拦截
            if torch.isnan(batch_X).any() or torch.isinf(batch_X).any(): continue
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            if torch.isnan(loss): continue
            
            loss.backward()
            # 防线二：安全阀梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item() * batch_X.size(0)
            correct_preds += ((outputs > 0).float() == batch_y).sum().item()
            total_samples += batch_y.size(0)
            valid_batches += 1
        del X_chunk, y_chunk, dataloader
        
    if total_samples > 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {total_loss/total_samples:.4f} | 截面预判胜率: {correct_preds/total_samples*100:.2f}% | 有效 Batch: {valid_batches}")

# 保存带有注意力模块的权重
torch.save(model.state_dict(), 'tcn_attention_weights.pth')
print("🏆 炼丹成功！新模型灵魂已封存至：tcn_attention_weights.pth")